In [1]:
import pandas as pd
import pickle
import numpy as np

from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)


In [2]:
# ============================================================
# 1. PARÂMETROS DO MLP
# ============================================================

MLP_PARAMS = {
    # Arquitetura da rede
    "hidden_layer_sizes": (64, 32, 16),

    # Função de ativação dos neurônios
    "activation": "relu",

    # Algoritmo de otimização
    "solver": "adam",

    # Taxa de aprendizado inicial
    "learning_rate_init": 0.001,

    # Quantidade máxima de épocas
    "max_iter": 100,

    # Tamanho do lote utilizado no treinamento
    "batch_size": 64,

    # Regularização L2
    "alpha": 0.0001,

    # Para interromper quando não houver melhoria
    "early_stopping": True,

    # Parte do treinamento utilizada para validação
    "validation_fraction": 0.1,

    # Número de épocas sem melhoria antes de parar
    "n_iter_no_change": 10,

    # Reprodutibilidade
    "random_state": 42
}

In [3]:
# ============================================================
# 2. CARREGAMENTO DO DATASET
# ============================================================

arquivo = "C:/Users/gques/Documents/SPTECH_CODIGOS/ultimo-ano/tcc/backend/tabela_final/part-00000-13fcc452-2493-4299-ab17-d369f64d2ad3-c000.csv"


df = pd.read_csv(arquivo)

In [4]:
# ============================================================
# 3. VARIÁVEL TARGET
# ============================================================

target = "delivered_on_time"

# ============================================================
# 4. SPLIT TEMPORAL
# ============================================================

split_date = pd.to_datetime("2018-07-01", utc=True)

date_col = pd.to_datetime(
    df["order_delivered_carrier_date"],
    utc=True
)

train_mask = date_col < split_date
test_mask = date_col >= split_date

df_train = df[train_mask].copy()
df_test = df[test_mask].copy()

print(f"Treino: {len(df_train):,} linhas")
print(f"Teste: {len(df_test):,} linhas")

# ============================================================
# 5. REMOÇÃO DE COLUNAS COM POSSÍVEL DATA LEAKAGE
# ============================================================

colunas_remover = [
    target,
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "interval_code_delivered_carrier",
    "order_delivered_customer_date",
    "interval_code_delivered_customer",
    "order_estimated_delivery_date",
    "shipping_limit_date",
    "is_holiday_in_7_days",
    "is_holiday_in_14_days",
    "had_holiday_7_days_ago",
    "had_holiday_14_days_ago",
    "customer_city",
    "seller_city",
    "delivered_on_time", 
    "category_name",
    "review_score",
    "delivery_time_days"
]

colunas_remover = [
    coluna for coluna in colunas_remover
    if coluna in df.columns
]

Treino: 28,706 linhas
Teste: 5,114 linhas


In [5]:
# ============================================================
# 6. SEPARAÇÃO ENTRE X E Y
# ============================================================

X_train = df_train.drop(columns=colunas_remover)
y_train = df_train[target]

X_test = df_test.drop(columns=colunas_remover)
y_test = df_test[target]


# ============================================================
# 7. CONVERSÃO DE VARIÁVEIS CATEGÓRICAS
# ============================================================

X_train = pd.get_dummies(
    X_train,
    drop_first=True
)

X_test = pd.get_dummies(
    X_test,
    drop_first=True
)

X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)

In [6]:
# ============================================================
# 8. TRATAMENTO DE VALORES AUSENTES
# ============================================================

X_train = X_train.replace(
    [np.inf, -np.inf],
    np.nan
)

X_test = X_test.replace(
    [np.inf, -np.inf],
    np.nan
)

X_train = X_train.fillna(
    X_train.median(numeric_only=True)
)

X_test = X_test.fillna(
    X_train.median(numeric_only=True)
)

X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

In [7]:
# ============================================================
# 9. NORMALIZAÇÃO
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [8]:
# ============================================================
# 10. CRIAÇÃO DO MLP
# ============================================================

mlp = MLPClassifier(
    **MLP_PARAMS
)


# ============================================================
# 11. TREINAMENTO
# ============================================================

mlp.fit(
    X_train_scaled,
    y_train
)


# ============================================================
# 11.1 SALVAMENTO DO MODELO
# ============================================================
modelo = {
    "model": mlp,
    "scaler": scaler,
    "features": X_train.columns.tolist()
}

caminho_modelo = "C:/Users/gques/Documents/SPTECH_CODIGOS/ultimo-ano/tcc/backend/modelo/dl/modelo_mlp_v1.pkl"

with open(caminho_modelo, "wb") as arquivo:
    pickle.dump(modelo, arquivo)

print(f"\nModelo salvo com sucesso em: {caminho_modelo}")


Modelo salvo com sucesso em: C:/Users/gques/Documents/SPTECH_CODIGOS/ultimo-ano/tcc/backend/modelo/dl/modelo_mlp_v1.pkl


In [9]:
# ============================================================
# 12. PREDIÇÃO
# ============================================================

y_pred = mlp.predict(
    X_test_scaled
)

y_prob = mlp.predict_proba(
    X_test_scaled
)[:, 1]

In [10]:
# ============================================================
# 13. MÉTRICAS
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    pos_label=0,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    pos_label=0,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    pos_label=0,
    zero_division=0
)

f2 = fbeta_score(
    y_test,
    y_pred,
    beta=2,
    pos_label=0,
    zero_division=0
)

auc = roc_auc_score(
    y_test,
    y_prob
)

In [11]:
# ============================================================
# 14. RESULTADOS
# ============================================================

print("\n" + "=" * 60)
print("RESULTADOS - MLP")
print("=" * 60)

print(f"Arquitetura: {MLP_PARAMS['hidden_layer_sizes']}")
print(f"Activation: {MLP_PARAMS['activation']}")
print(f"Learning rate: {MLP_PARAMS['learning_rate_init']}")
print(f"Batch size: {MLP_PARAMS['batch_size']}")
print(f"Épocas máximas: {MLP_PARAMS['max_iter']}")

print("\nQuantidade de amostras:")
print(f"Treino: {len(X_train)}")
print(f"Teste:  {len(X_test)}")

print("\nMétricas:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"F2-score:  {f2:.4f}")
print(f"ROC-AUC:   {auc:.4f}")

print("\nMatriz de confusão:")
print(confusion_matrix(y_test, y_pred))

print("\nRelatório de classificação:")
print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)

print("\nQuantidade de épocas executadas:")
print(mlp.n_iter_)


RESULTADOS - MLP
Arquitetura: (64, 32, 16)
Activation: relu
Learning rate: 0.001
Batch size: 64
Épocas máximas: 100

Quantidade de amostras:
Treino: 28706
Teste:  5114

Métricas:
Accuracy:  0.8774
Precision: 0.0625
Recall:    0.0016
F1-score:  0.0032
F2-score:  0.0020
ROC-AUC:   0.4735

Matriz de confusão:
[[   1  612]
 [  15 4486]]

Relatório de classificação:
              precision    recall  f1-score   support

           0       0.06      0.00      0.00       613
           1       0.88      1.00      0.93      4501

    accuracy                           0.88      5114
   macro avg       0.47      0.50      0.47      5114
weighted avg       0.78      0.88      0.82      5114


Quantidade de épocas executadas:
21


Esse resultado mostra um problema bem claro no MLP: **ele praticamente não está aprendendo a identificar os atrasos**.

Pelos números:

```text
Matriz de confusão:

[[   1  612]
 [  15 4486]]
```

Considerando `0 = atraso` e `1 = no prazo`:

|                   | Predito atraso | Predito no prazo |
| ----------------- | -------------- | ---------------- |
| **Real atraso**   | 1              | 612              |
| **Real no prazo** | 15             | 4486             |

Ou seja:

* **613 entregas estavam atrasadas**
* O modelo identificou apenas **1**
* Deixou passar **612 atrasos**
* Identificou **4.486 entregas no prazo** corretamente

Isso explica:

```text
Accuracy: 0.8774
Recall:   0.0016
F1:       0.0032
F2:       0.0020
```

A accuracy de **87,74% parece boa**, mas é enganosa nesse problema porque a classe "no prazo" é muito maior.

### O problema aparece claramente no ROC-AUC

```text
ROC-AUC: 0.4735
```

Um modelo com capacidade discriminativa razoável deveria ficar acima de 0,5; **0,4735 indica que, com essa configuração, o MLP não está conseguindo separar adequadamente as duas classes**.

E o fato de ele ter parado em:

```text
21 épocas
```

não é necessariamente um erro. Você habilitou:

```python
"early_stopping": True,
"validation_fraction": 0.1,
"n_iter_no_change": 10,
```

Então o `MLPClassifier` interrompeu o treinamento quando não encontrou melhoria suficiente na validação.

---

## O principal problema provavelmente é o desbalanceamento

Seu teste tem:

```text
Atraso:    613
No prazo: 4501
```

Ou aproximadamente:

```text
12% atraso
88% no prazo
```

Então o modelo encontra uma estratégia muito fácil:

> "Quase tudo está no prazo."

E isso produz aproximadamente os 88% de accuracy que você está vendo.

Isso é particularmente importante no seu TCC porque você **não quer apenas prever a classe majoritária**. O objetivo é justamente identificar os pedidos que irão atrasar.

---

## Tem outro ponto importante no seu código

Você está calculando:

```python
precision_score(
    y_test,
    y_pred,
    pos_label=0
)
```

Isso está correto **se no seu projeto `0 = atraso`**.

E:

```python
recall_score(..., pos_label=0)
```

também está correto.

Então seus indicadores estão realmente medindo a capacidade de detectar **atrasos**, que é o que interessa.
